# Atividade Prática — Aula 2: Pandas Essencial

Nesta atividade, você vai aplicar os conceitos da aula para explorar um dataset de vendas, fazer um **check-up inicial**, selecionar colunas relevantes, criar **filtros de negócio**, construir **rankings** e transformar resultados em **interpretação gerencial**.

**Dataset:** `vendas_dataviz_aula2.csv`

## Objetivos da atividade
- Ler e inspecionar um dataset com Pandas
- Entender a diferença entre DataFrame e Series
- Verificar tamanho, tipos e possíveis problemas de qualidade
- Selecionar apenas as colunas importantes para análise
- Filtrar dados com uma e múltiplas condições
- Criar rankings com `sort_values`, `nlargest` e `nsmallest`
- Interpretar resultados em linguagem de negócio


## 1. Importação das bibliotecas

Importe as bibliotecas necessárias para a atividade.
- `pandas`
- `numpy`
- `matplotlib.pyplot` (caso queira visualizar resultados)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Leitura do dataset

Leia o arquivo `vendas_dataviz_aula2.csv` em um DataFrame chamado `df`.
Depois, exiba as 5 primeiras linhas.


In [ ]:
df = pd.read_csv('vendas_dataviz_aula2.csv')
df.head()

## 3. Anatomia dos dados: DataFrame e Series

### Questão 1
Explique, com base no que foi visto em aula:
1. O que é um **DataFrame**?
2. O que é uma **Series**?
3. Mostre no código uma coluna isolada do DataFrame.


In [ ]:
# Exemplo: selecione uma única coluna e observe o tipo retornado
serie_produto = df['produto']
type(serie_produto), serie_produto.head()

## 4. Check-up inicial do dataset

Um analista profissional sempre começa verificando o tamanho, a estrutura e a qualidade inicial dos dados.

### Questão 2
Use:
- `df.shape`
- `df.info()`
- `df.dtypes`

Depois responda:
1. Quantas linhas e colunas existem?
2. Quais colunas parecem numéricas?
3. Há sinais de valores ausentes?
4. Existe alguma coluna com tipo inadequado?


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

## 5. Inspeção de problemas de qualidade

Na aula, vimos que dashboards podem falhar por causa de dados mal preparados.

### Questão 3
Investigue:
- valores nulos
- categorias inconsistentes
- números armazenados como texto
- valores infinitos

Dica:
- `df.isna().sum()`
- `df['canal'].value_counts(dropna=False)`
- `df['estado'].value_counts(dropna=False)`
- `np.isinf(...)`


In [ ]:
df.isna().sum()

In [ ]:
df['canal'].value_counts(dropna=False)

In [ ]:
df['estado'].value_counts(dropna=False)

In [ ]:
# Verifique se há valores infinitos na coluna lucro
pd.Series(np.isinf(pd.to_numeric(df['lucro'], errors='coerce'))).value_counts(dropna=False)

## 6. Selecionando o que importa

Nem toda análise precisa de todas as colunas. Vamos criar um recorte mais focado para responder perguntas de negócio.

### Questão 4
Crie um novo DataFrame chamado `df_dash` contendo apenas as colunas:
- `data`
- `estado`
- `canal`
- `produto`
- `categoria`
- `vendas`
- `lucro`

Use `.copy()` para evitar problemas futuros.


In [ ]:
cols_dashboard = ['data', 'estado', 'canal', 'produto', 'categoria', 'vendas', 'lucro']
df_dash = df[cols_dashboard].copy()
df_dash.head()

## 7. Preparação mínima para análise

Algumas colunas podem precisar de ajuste antes de ordenar e agregar.

### Questão 5
1. Converta a coluna `data` para formato de data.
2. Crie uma coluna `mes` a partir da data.
3. Tente converter `vendas` para numérico, tratando possíveis textos.
4. Crie uma coluna chamada `vendas_num`.


In [ ]:
df_dash['data'] = pd.to_datetime(df_dash['data'], errors='coerce')
df_dash['mes'] = df_dash['data'].dt.to_period('M')

# Limpeza simples para a coluna vendas (remove símbolo R$, espaços e troca vírgula decimal)
vendas_limpa = (
    df_dash['vendas']
    .astype(str)
    .str.replace('R\$', '', regex=True)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

df_dash['vendas_num'] = pd.to_numeric(vendas_limpa, errors='coerce')
df_dash.head()

## 8. Filtragem simples: recorte de negócio

Na aula, vimos o exemplo de filtrar apenas o estado do Rio de Janeiro.

### Questão 6
Crie um DataFrame `df_rj` contendo apenas registros do estado `RJ`.
Depois responda:
1. Quantos registros existem nesse recorte?
2. Qual a soma de vendas no RJ?


In [ ]:
df_rj = df_dash[df_dash['estado'] == 'RJ']
df_rj.shape, df_rj['vendas_num'].sum()

## 9. Filtragem com múltiplas condições

Agora faça um recorte mais profissional: apenas vendas do estado do RJ no canal Online.

### Questão 7
Faça esse filtro de duas formas:
1. Usando operadores lógicos com `&`
2. Usando `query()`


In [ ]:
df_rj_online = df_dash[(df_dash['estado'] == 'RJ') & (df_dash['canal'] == 'Online')]
df_rj_online.head()

In [ ]:
df_rj_online_q = df_dash.query("estado == 'RJ' and canal == 'Online'")
df_rj_online_q.head()

## 10. Priorização: rankings

Gestores raramente leem tabelas enormes. Eles querem os **Top N**.

### Questão 8
Gere:
1. Os 10 registros com maior valor de `vendas_num`
2. Os 10 registros com maior `lucro`
3. Os 5 registros com menor `lucro`

Observe se aparece algum valor suspeito.


In [ ]:
top10_vendas = df_dash.sort_values('vendas_num', ascending=False).head(10)
top10_vendas

In [ ]:
# Primeiro garantimos que lucro esteja numérico
df_dash['lucro_num'] = pd.to_numeric(df_dash['lucro'], errors='coerce')

top10_lucro = df_dash.nlargest(10, 'lucro_num')
top10_lucro

In [ ]:
bottom5_lucro = df_dash.nsmallest(5, 'lucro_num')
bottom5_lucro

## 11. Ranking por produto

Agora vamos sair do nível do registro e pensar em **resultado por produto**.

### Questão 9
Agrupe por produto e calcule:
- soma de vendas
- soma de lucro

Depois:
1. Mostre os 10 produtos mais lucrativos
2. Interprete o resultado em linguagem de negócio


In [ ]:
ranking_produtos = (
    df_dash.groupby('produto', dropna=False)
    .agg(
        vendas_total=('vendas_num', 'sum'),
        lucro_total=('lucro_num', 'sum')
    )
    .sort_values('lucro_total', ascending=False)
)

ranking_produtos.head(10)

## 12. Ranking por estado e canal

### Questão 10
Crie dois rankings:
1. Estados com maior receita total
2. Canais com maior lucro total

Depois escreva uma interpretação curta para cada ranking.


In [ ]:
ranking_estados = (
    df_dash.groupby('estado', dropna=False)['vendas_num']
    .sum()
    .sort_values(ascending=False)
)

ranking_canais = (
    df_dash.groupby('canal', dropna=False)['lucro_num']
    .sum()
    .sort_values(ascending=False)
)

ranking_estados, ranking_canais

## 13. Análise temporal simples

### Questão 11
Agrupe os dados por `mes` e calcule a receita total mensal.
Depois responda:
1. Há meses de maior desempenho?
2. Existe indício de sazonalidade?


In [ ]:
receita_mensal = (
    df_dash.groupby('mes')['vendas_num']
    .sum()
    .sort_index()
)

receita_mensal

In [ ]:
receita_mensal.plot(kind='bar', figsize=(12,4), title='Receita mensal')
plt.ylabel('Receita')
plt.show()

## 14. A última milha: interpretação

Na aula, vimos que um analista não entrega apenas tabela — entrega **decisão**.

### Questão 12
Com base nos resultados do notebook, escreva interpretações gerenciais para:
1. O produto mais lucrativo
2. O estado com maior receita
3. O canal com maior lucro
4. Um possível problema de qualidade encontrado


## 15. Desafio extra (opcional)

Faça uma limpeza adicional do dataset:
- padronize valores de `canal` (`online`, `ONLINE`, `MarketPlace`, etc.)
- padronize `estado` (`rj` -> `RJ`)
- substitua infinitos em `lucro_num` por `NaN`
- trate valores ausentes como julgar adequado

Depois gere novamente os rankings e compare os resultados.


In [ ]:
# Escreva sua solução aqui


## 16. Entrega esperada

Seu notebook deve demonstrar:
- organização
- comentários explicativos
- código legível
- interpretação de negócio

### Fechamento
Ao terminar, salve o notebook com suas respostas e envie junto com o arquivo CSV utilizado.
